In [1]:

import os
# --- CPU OPTIMIZATION FLAGS ---
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '1' # Intel optimization for CPU
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress generic TF info logs


In [2]:

import tensorflow as tf
from tensorflow.keras import layers, models, applications, optimizers, callbacks
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report


In [3]:

# --- 1. CONFIGURATION & PATHS ---
BASE_DIR = Path('../Data/dataset_10')
TRAIN_DIR = BASE_DIR / 'train'
VAL_DIR   = BASE_DIR / 'val'
TEST_DIR  = BASE_DIR / 'test'
PRED_DIR  = BASE_DIR / 'single_prediction'

IMG_SIZE = 224 
BATCH_SIZE = 16
EPOCHS = 5


In [4]:

# --- 2. DATA PIPELINE & IMBALANCE HANDLING ---
# 2a. Raw Dataset Loading
def get_dataset(directory, shuffle=True):
    return tf.keras.utils.image_dataset_from_directory(
        directory,
        seed=123,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=None,
        label_mode='int',
        shuffle=shuffle
    )

train_ds_raw = get_dataset(TRAIN_DIR, shuffle=True)
val_ds_raw = get_dataset(VAL_DIR, shuffle=False)
test_ds_raw = get_dataset(TEST_DIR, shuffle=False)

class_names = train_ds_raw.class_names
print(f'Classes found: {class_names}')

# 2b. Manual Class Weights Calculation
class_counts = {}
total_samples = 0

for cls_idx, cls_name in enumerate(class_names):
    cls_path = TRAIN_DIR / cls_name
    if cls_path.exists():
        # Count only files
        files = [f for f in os.listdir(cls_path) if os.path.isfile(os.path.join(cls_path, f))]
        count = len(files)
        class_counts[cls_idx] = count
        total_samples += count
    else:
        class_counts[cls_idx] = 0

class_weights = {}
for cls_idx, count in class_counts.items():
    if count > 0:
        weight = total_samples / (len(class_names) * count)
        class_weights[cls_idx] = weight
    else:
        class_weights[cls_idx] = 1.0

print(f'Class Weights: {class_weights}')

# 2c. Optimized Pipeline Logic
# Define augmentation layers
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

def preprocess(x, y):
    return tf.cast(x, tf.float32), y

def augment(x, y):
    return data_augmentation(x), y

AUTOTUNE = tf.data.AUTOTUNE

def prepare_dataset(ds, is_train=False):
    ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()

    if is_train:
        ds = ds.shuffle(1000)
        ds = ds.batch(BATCH_SIZE)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.batch(BATCH_SIZE)

    ds = ds.prefetch(buffer_size=AUTOTUNE)
    return ds

train_ds = prepare_dataset(train_ds_raw, is_train=True)
val_ds = prepare_dataset(val_ds_raw, is_train=False)
test_ds = prepare_dataset(test_ds_raw, is_train=False)


Found 20938 files belonging to 10 classes.
Found 2614 files belonging to 10 classes.
Found 2627 files belonging to 10 classes.
Classes found: ['butterfly', 'cat', 'chicken', 'cow', 'dog', 'elephant', 'horse', 'ragno', 'sheep', 'squirrel']
Class Weights: {0: 1.2396684428656009, 1: 1.5695652173913044, 2: 0.844955609362389, 3: 1.4033512064343163, 4: 0.5382519280205655, 5: 1.8112456747404844, 6: 0.9979980934223069, 7: 0.5429979253112033, 8: 1.4380494505494505, 9: 1.406178643384822}


In [5]:

# --- 3. HYBRID MODEL (CNN + LSTM) ---
def build_hybrid_model(num_classes):
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    # CNN Backbone (EfficientNetB0)
    base_model = applications.EfficientNetB0(
        include_top=False, 
        weights='imagenet', 
        input_tensor=inputs
    )
    base_model.trainable = False 

    x = base_model.output # Shape: (Batch, 7, 7, 1280) for IMG_SIZE=224

    # Reshape for RNN: Flatten spatial dims (7*7) into sequence steps (49)
    x = layers.Reshape((49, 1280))(x)

    # RNN Component (Bidirectional LSTM)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=False))(x)

    # ANN Component
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs, name='Hybrid_CNN_LSTM')

model = build_hybrid_model(len(class_names))
model.summary()


Model: "Hybrid_CNN_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 5,560,749 (21.21 MB)

 Trainable params: 1,511,178 (5.76 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:

# --- 4. TRAINING ---
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.995e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', mode='min', patience=5, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', mode='min', factor=0.2, patience=3, min_lr=1e-6)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights
)


Epoch 1/5
1309/1309 ━━━━━━━━━━━━━━━━━━━━ 1100s 831ms/step - accuracy: 0.8759 - loss: 0.4128 - val_accuracy: 0.9487 - val_loss: 0.1680 - learning_rate: 9.9500e-04
Epoch 2/5
   5/1309 ━━━━━━━━━━━━━━━━━━━━ 23:12 1s/step - accuracy: 0.8615 - loss: 0.3255   

In [ ]:

# 5. Visualize the training and validation of the model
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Accuracy
axes[0].plot(acc, label='Training Acc', color='#1f77b4')
axes[0].plot(val_acc, label='Validation Acc', color='#ff7f0e')
axes[0].set_xticks(np.arange(0, EPOCHS))
axes[0].set_xticklabels([str(_) for _ in range(1, EPOCHS+1)])
axes[0].set_yticks(np.arange(0, 1.01, 0.05))
axes[0].legend(loc='lower right')
axes[0].set_title('Training and Validation Accuracy')
axes[0].grid(True, alpha=0.5)

# Plot 2: Loss
axes[1].plot(loss, label='Training Loss', color='#d62728')
axes[1].plot(val_loss, label='Validation Loss', color='#2ca02c')
axes[1].set_xticks(np.arange(0, EPOCHS))
axes[1].set_xticklabels([str(_) for _ in range(1, EPOCHS+1)])
axes[1].set_yticks(np.arange(0, max(max(loss), max(val_loss)) + 0.02, 0.02))
axes[1].legend(loc='upper right')
axes[1].set_title('Training and Validation Loss')
axes[1].grid(True, alpha=0.5)

plt.tight_layout()
plt.savefig('training_plots.png')
plt.show()


In [ ]:

y_true = []
y_pred_probs = []

for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    y_pred_probs.extend(probs)
    y_true.extend(labels.numpy())

y_pred = np.argmax(y_pred_probs, axis=1)


In [ ]:

# Plot Confusion Matrix
plt.figure(figsize=(12, 10))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='tab10', 
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, linecolor='white', square=True)
plt.title('Confusion Matrix', fontsize=16)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


In [ ]:

# Testing Classification Report
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)


In [ ]:

# --- 6. FINAL EVALUATION ON TEST SET ---
print('--- Evaluation on Test Set ---')
test_loss, test_acc = model.evaluate(test_ds)
print(f'Final Test Accuracy: {test_acc * 100:.4f}%')


In [ ]:

# --- 7. SINGLE PREDICTION TASK ---
print('--- Single Prediction Task ---')
images = os.listdir(PRED_DIR)

for img_file in images:
	img_path = PRED_DIR / img_file

	img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
	img_array = img_to_array(img)
	img_array = tf.expand_dims(img_array, 0)

	img_array = tf.cast(img_array, tf.float32)

	# Predict
	score = model.predict(img_array)
	predicted_class = class_names[np.argmax(score)]
	confidence = 100 * np.max(score)

	print(f'Image: {img_file} | Predicted: {predicted_class} | Confidence: {confidence:.2f}%')
